# NMNIST — Surrogate Gradient Scaling Study

Train a fully-connected LIF SNN on NMNIST at different network widths and
temporal resolutions.  See how accuracy, spike rate, and training time scale
with (channels × timesteps).  Compare with the equivalent SHD study.

In [ ]:
import spyx
import spyx.nn as snn

import jax
from jax import numpy as jnp
import haiku as hk
import optax
import jmp
import numpy as np

from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
%matplotlib notebook

In [ ]:
policy = jmp.get_policy('half')
hk.mixed_precision.set_policy(hk.Linear, policy)
hk.mixed_precision.set_policy(snn.LIF, policy)
hk.mixed_precision.set_policy(snn.LI, policy)

### NMNIST Dataloading

In [ ]:
import tonic
from tonic import datasets, transforms
from torch.utils.data import DataLoader

sensor_size = tonic.datasets.NMNIST.sensor_size  # (34, 34, 2)
n_classes  = 10

# Configs to sweep: (channels, timesteps)
CONFIGS = [
    (64,  64),
    (64,  128),
    (64,  256),
    (64,  512),
    (128, 64),
    (128, 128),
    (128, 256),
    (128, 512),
    (256, 64),
    (256, 128),
    (256, 256),
    (256, 512),
    (512, 64),
    (512, 128),
    (512, 256),
    (512, 512),
]

In [ ]:
def load_nmnist(sample_T, net_channels):
    """Load and rasterise NMNIST at the given temporal and spatial resolution."""
    frame_transform = transforms.Compose([
        transforms.ToFrame(sensor_size=sensor_size, n_time_bins=sample_T),
        lambda x: np.minimum(x, 1),
        lambda x: np.packbits(x, axis=0)
    ])
    train_ds = tonic.datasets.NMNIST(
        save_to='./data', transform=frame_transform, train=True
    )
    test_ds  = tonic.datasets.NMNIST(
        save_to='./data', transform=frame_transform, train=False
    )

    train_dl = iter(DataLoader(train_ds, batch_size=2048,
                              collate_fn=tonic.collation.PadTensors(batch_first=True),
                              drop_last=True, shuffle=False))
    test_dl  = iter(DataLoader(test_ds,  batch_size=1024,
                              collate_fn=tonic.collation.PadTensors(batch_first=True),
                              drop_last=True, shuffle=False))

    x_train, y_train = next(train_dl)
    x_test,  y_test  = next(test_dl)

    return (
        jnp.array(x_train, dtype=jnp.uint8),
        jnp.array(y_train, dtype=jnp.uint8),
        jnp.array(x_test,  dtype=jnp.uint8),
        jnp.array(y_test,  dtype=jnp.uint8),
    )

In [ ]:
surrogate = spyx.axn.Axon(spyx.axn.arctan())

def build_snn(n_input, n_classes, channels, sample_T):
    """2-layer FC SNN: Linear→LIF→Linear→LIF→Linear→LI."""
    def snn_fn(x):
        x = hk.BatchApply(hk.Linear(channels, with_bias=False))(x)
        core = hk.DeepRNN([
            snn.LIF((channels,), activation=surrogate),
            hk.Linear(channels, with_bias=False),
            snn.LIF((channels,), activation=surrogate),
            hk.Linear(n_classes, with_bias=False),
            snn.LI((n_classes,))
        ])
        spikes, V = hk.dynamic_unroll(
            core, x, core.initial_state(x.shape[0]),
            time_major=False, unroll=16
        )
        return spikes, V
    return hk.without_apply_rng(hk.transform(snn_fn))

In [ ]:
@jax.jit
def net_eval(weights, events, targets):
    traces, _ = SNN.apply(weights, events)
    return spyx.fn.integral_crossentropy(traces, targets, smoothing=0.2)

surrogate_grad = jax.value_and_grad(net_eval)

@jax.jit
def train_step(state, batch):
    weights, opt_state = state
    events, targets = batch
    events = jnp.unpackbits(events, axis=1)
    loss, grads = surrogate_grad(weights, events, targets)
    updates, opt_state = opt.update(grads, opt_state, weights)
    weights = optax.apply_updates(weights, updates)
    return (weights, opt_state), loss

@jax.jit
def eval_step(weights, batch):
    events, targets = batch
    events = jnp.unpackbits(events, axis=1)
    traces, _ = SNN.apply(weights, events)
    acc, _ = spyx.fn.integral_accuracy(traces, targets)
    loss = net_eval(weights, events, targets)
    return jnp.array([acc, loss])

In [ ]:
EPOCHS = 100
LR     = 2e-4
opt    = optax.chain(optax.centralize(), optax.lion(learning_rate=LR))

results = []

for channels, sample_T in CONFIGS:
    print(f"\n{'='*50}")
    print(f"  NMNIST  channels={channels}  timesteps={sample_T}")
    print(f"{'='*50}")

    x_train, y_train, x_test, y_test = load_nmnist(sample_T, channels)
    n_input = int(np.prod(x_train.shape[2:]))   # flattened sensor size

    SNN = build_snn(n_input, n_classes, channels, sample_T)
    key = jax.random.PRNGKey(0)
    params = SNN.init(rng=key, x=x_train[:32])
    opt_state = opt.init(params)

    n_train = y_train.shape[0]
    indices = jnp.arange(n_train)
    BATCH = 128

    best_acc = 0.0
    rng = hk.next_rng_key()()

    for ep in tqdm(range(EPOCHS), desc=f"{channels}c-{sample_T}t"):
        rng, = jax.random.split(rng)
        perm = jax.random.permutation(rng, indices)

        for i in range(0, n_train - BATCH + 1, BATCH):
            batch_idx = perm[i:i+BATCH]
            (params, opt_state), _ = train_step(
                (params, opt_state),
                (x_train[batch_idx], y_train[batch_idx])
            )

        acc, loss = eval_step(params, (x_test, y_test))
        acc = float(acc)
        if acc > best_acc:
            best_acc = acc

    n_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
    print(f"  → Best accuracy: {best_acc:.2%}  |  params: {n_params:,}")
    results.append(dict(channels=channels, timesteps=sample_T,
                        accuracy=best_acc, n_params=n_params))

print("\nDone.")

### Results heatmap

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
pivot = df.pivot(index='channels', columns='timesteps', values='accuracy')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values * 100, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('Timesteps'); ax.set_ylabel('Channels')
ax.set_title('NMNIST accuracy (%) — channels × timesteps')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]*100:.1f}",
                ha='center', va='center', color='white', fontsize=9)
plt.colorbar(im, ax=ax, label='Accuracy (%)')
plt.tight_layout()
plt.show()

print(df.sort_values('accuracy', ascending=False).to_string(index=False))